# 🏋️ PoseCoach P01 — Dataset Prep & YOLO26 Finetune
> **Run on:** Google Colab T4 GPU | **Storage:** Your Google Drive
>
> Based on [Ultralytics YOLO26 Fine-Tuning Guide](https://docs.ultralytics.com/guides/finetuning-guide/)
>
> Uses **two-stage training**: freeze backbone → unfreeze all with lower LR

### Steps
1. Mount Google Drive & verify GPU
2. Upload Kaggle credentials
3. Install dependencies
4. Download dataset
5. Extract frames at 1 FPS
6. Extract YOLO26 keypoints
7. Prepare YOLO pose dataset (80/20 stratified split)
8. Baseline evaluation
9. **Stage 1** — Freeze backbone, train head & neck (20 epochs)
10. **Stage 2** — Unfreeze all, fine-tune with lower LR (30 epochs)
11. Save weights & evaluate

## ✅ Step 1 — Verify GPU & Mount Google Drive

In [ ]:
import torch, os
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — switch runtime to T4!'}")
print(f"CUDA: {torch.version.cuda}")

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/GYMVISION AI'
DATASET_DIR = f'{DRIVE_ROOT}/datasets/workoutfitness'
FRAMES_DIR  = f'{DRIVE_ROOT}/datasets/frames'
KP_DIR      = f'{DRIVE_ROOT}/datasets/keypoints'
YOLO_DIR    = f'{DRIVE_ROOT}/datasets/yolo_pose'
MODELS_DIR  = f'{DRIVE_ROOT}/models'

for d in [DATASET_DIR, FRAMES_DIR, KP_DIR, YOLO_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print('\n✅ Google Drive mounted and folders created')
print(f'   Root: {DRIVE_ROOT}')

## ✅ Step 2 — Upload Kaggle API Key

In [ ]:
# Get it from: https://www.kaggle.com/settings → API → Create New Token
from google.colab import files
print('Upload your kaggle.json file now...')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials configured')

## ✅ Step 3 — Install Dependencies

In [ ]:
%%capture
!pip install ultralytics kaggle opencv-python-headless numpy tqdm

## ✅ Step 4 — Download Dataset from Kaggle

In [ ]:
import subprocess, sys

DATASET_SLUG = 'hasyimabdillah/workoutfitnessvideo'

if os.path.exists(f'{DATASET_DIR}/videos') and len(os.listdir(f'{DATASET_DIR}/videos')) > 0:
    print('✅ Dataset already downloaded — skipping')
else:
    print(f'Downloading {DATASET_SLUG}...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', DATASET_SLUG, '-p', DATASET_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('ERROR:', result.stderr)
        sys.exit(1)

    print('Unzipping...')
    import zipfile
    for zf in [f for f in os.listdir(DATASET_DIR) if f.endswith('.zip')]:
        with zipfile.ZipFile(f'{DATASET_DIR}/{zf}', 'r') as z:
            z.extractall(DATASET_DIR)
        os.remove(f'{DATASET_DIR}/{zf}')
    print('✅ Dataset downloaded and extracted')

for root, dirs, files_list in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    if level < 2:
        print('  ' * level + os.path.basename(root) + '/')
        if level == 1:
            print('  ' * (level+1) + f'({len(files_list)} files)')

## ✅ Step 5 — Extract Frames at 1 FPS

In [ ]:
import cv2
from tqdm import tqdm
from pathlib import Path

def extract_frames(video_path, out_dir, fps=1):
    cap = cv2.VideoCapture(video_path)
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    interval = max(1, int(video_fps / fps))
    os.makedirs(out_dir, exist_ok=True)
    count = saved = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if count % interval == 0:
            cv2.imwrite(f'{out_dir}/frame_{saved:06d}.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved += 1
        count += 1
    cap.release()
    return saved

video_exts = {'.mp4', '.avi', '.mov', '.mkv'}
video_files = [p for p in Path(DATASET_DIR).rglob('*') if p.suffix.lower() in video_exts]
print(f'Found {len(video_files)} video files')

total_frames = 0
for vid in tqdm(video_files, desc='Extracting frames'):
    clip_id = vid.stem
    exercise = vid.parent.name
    out = f'{FRAMES_DIR}/{exercise}/{clip_id}'
    if os.path.exists(out) and len(os.listdir(out)) > 0:
        total_frames += len(os.listdir(out))
        continue
    total_frames += extract_frames(str(vid), out, fps=1)

print(f'\n✅ Extracted {total_frames:,} frames total')
for ex in os.listdir(FRAMES_DIR):
    clips = os.listdir(f'{FRAMES_DIR}/{ex}')
    frames = sum(len(os.listdir(f'{FRAMES_DIR}/{ex}/{c}')) for c in clips)
    print(f'  {ex}: {len(clips)} clips, {frames} frames')

## ✅ Step 6 — Extract YOLO26 Keypoints

In [ ]:
import numpy as np
from ultralytics import YOLO

# YOLO26n-pose — NMS-free end-to-end (DO NOT add NMS after predict)
model = YOLO('yolo26n-pose.pt')
model.to('cuda' if torch.cuda.is_available() else 'cpu')

frame_paths = list(Path(FRAMES_DIR).rglob('*.jpg'))
print(f'Processing {len(frame_paths):,} frames for keypoints...')

skipped = saved = 0
for img_path in tqdm(frame_paths, desc='Extracting keypoints'):
    rel = img_path.relative_to(FRAMES_DIR)
    kp_path = Path(KP_DIR) / rel.with_suffix('.npy')
    kp_path.parent.mkdir(parents=True, exist_ok=True)
    if kp_path.exists():
        skipped += 1
        continue
    results = model.predict(str(img_path), verbose=False, conf=0.3)
    if results[0].keypoints is None or len(results[0].keypoints) == 0:
        continue
    # CRITICAL: use .xyn (normalized) NOT .xy
    kp = results[0].keypoints.xyn.cpu().numpy()
    np.save(str(kp_path), kp)
    saved += 1

print(f'\n✅ Keypoints saved: {saved:,} | Skipped (cached): {skipped:,}')

## ✅ Step 7 — Prepare YOLO Pose Dataset (80/20 Stratified Split)

In [ ]:
import shutil, random, yaml
from collections import defaultdict

random.seed(42)

EXERCISE_MAP = {
    'squat': 0, 'deadlift': 1, 'curl': 2,
    'bench': 3, 'ohp': 4, 'lunge': 5, 'plank': 6
}

clips_by_exercise = defaultdict(list)
for img_path in Path(FRAMES_DIR).rglob('*.jpg'):
    exercise = img_path.parent.parent.name.lower()
    clip_id = img_path.parent.name
    matched = next((k for k in EXERCISE_MAP if k in exercise), None)
    if matched:
        clips_by_exercise[matched].append(clip_id)

for ex in clips_by_exercise:
    clips_by_exercise[ex] = list(set(clips_by_exercise[ex]))

train_clips, val_clips = set(), set()
for ex, clips in clips_by_exercise.items():
    random.shuffle(clips)
    split = max(1, int(len(clips) * 0.8))
    train_clips.update(clips[:split])
    val_clips.update(clips[split:])
    print(f'  {ex}: {split} train, {len(clips)-split} val clips')

for split_name, clip_set in [('train', train_clips), ('val', val_clips)]:
    img_out = Path(YOLO_DIR) / split_name / 'images'
    lbl_out = Path(YOLO_DIR) / split_name / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    for img_path in Path(FRAMES_DIR).rglob('*.jpg'):
        clip_id = img_path.parent.name
        if clip_id not in clip_set: continue
        exercise = img_path.parent.parent.name.lower()
        matched = next((k for k in EXERCISE_MAP if k in exercise), None)
        if not matched: continue
        rel = img_path.relative_to(FRAMES_DIR)
        kp_path = Path(KP_DIR) / rel.with_suffix('.npy')
        if not kp_path.exists(): continue
        kp = np.load(str(kp_path))
        if kp.shape[0] == 0: continue
        kp = kp[0]
        valid = kp[kp[:, 0] > 0]
        if len(valid) < 5: continue
        x1, y1 = valid[:, 0].min(), valid[:, 1].min()
        x2, y2 = valid[:, 0].max(), valid[:, 1].max()
        cx = (x1+x2)/2; cy = (y1+y2)/2
        w = (x2-x1)*1.2; h = (y2-y1)*1.2
        cx,cy,w,h = [min(max(v,0),1) for v in [cx,cy,w,h]]
        kp_flat = ' '.join(f'{kp[i,0]:.6f} {kp[i,1]:.6f} 2' for i in range(17))
        label_line = f'{EXERCISE_MAP[matched]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f} {kp_flat}'
        uid = f'{clip_id}_{img_path.stem}'
        shutil.copy(str(img_path), str(img_out / f'{uid}.jpg'))
        (lbl_out / f'{uid}.txt').write_text(label_line)

yaml_content = {
    'path': YOLO_DIR, 'train': 'train/images', 'val': 'val/images',
    'nc': 7, 'names': list(EXERCISE_MAP.keys()), 'kpt_shape': [17, 3]
}
yaml_path = f'{YOLO_DIR}/dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)

train_n = len(list((Path(YOLO_DIR)/'train'/'images').glob('*.jpg')))
val_n = len(list((Path(YOLO_DIR)/'val'/'images').glob('*.jpg')))
print(f'\n✅ Dataset ready: {train_n} train | {val_n} val images')

## ✅ Step 8 — Baseline Evaluation (Pretrained YOLO26n-Pose)

In [ ]:
import json, time

baseline_model = YOLO('yolo26n-pose.pt')
print('Running baseline evaluation on val set...')
metrics = baseline_model.val(data=yaml_path, split='val', verbose=False)

baseline_results = {
    'model': 'yolo26n-pose-pretrained',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'pose_map50': float(metrics.pose.map50) if hasattr(metrics, 'pose') else 0.0,
    'pose_map': float(metrics.pose.map) if hasattr(metrics, 'pose') else 0.0,
}

os.makedirs(f'{DRIVE_ROOT}/data/eval', exist_ok=True)
with open(f'{DRIVE_ROOT}/data/eval/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print(f"\n✅ Baseline OKS-mAP@0.50: {baseline_results['pose_map50']:.4f}")
print(f"   Thesis target after finetuning: >= 0.75")

## ✅ Step 9 — Two-Stage Finetune YOLO26n-Pose 🚀

> Per [Ultralytics YOLO26 Guide](https://docs.ultralytics.com/guides/finetuning-guide/#two-stage-fine-tuning):
>
> **Stage 1:** `freeze=10` (backbone frozen) — head & neck learn gym exercises
>
> **Stage 2:** `freeze=None` (all unfrozen) — full model adapts with lower LR
>
> Key settings:
> - `optimizer='AdamW'` explicit (auto overrides lr0!)
> - `mosaic=0.5` (reduced — heavy augmentation hurts small datasets)
> - `patience=15` (guide recommends 10-20)
> - YOLO26 is NMS-free end-to-end — NEVER add NMS after predict

In [ ]:
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'

# ═══════════════════════════════════════════
# STAGE 1: Freeze backbone, train head & neck
# ═══════════════════════════════════════════
print('🔒 STAGE 1: Frozen backbone — training head & neck only')
model = YOLO('yolo26n-pose.pt')  # YOLO26 — NMS-free end-to-end

model.train(
    data=yaml_path,
    epochs=20,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project=f'{MODELS_DIR}/runs',
    name='posecoach_stage1',
    exist_ok=True,
    freeze=10,           # freeze backbone (layers 0-9)
    optimizer='AdamW',   # explicit — 'auto' overrides lr0
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    patience=15,
    save=True,
    save_period=5,
    pose=12.0,           # higher pose loss weight
    mosaic=0.5,          # reduced — heavy aug hurts small datasets
    verbose=True,
)
print('✅ Stage 1 complete!')

In [ ]:
# ═══════════════════════════════════════════
# STAGE 2: Unfreeze all, fine-tune full model
# ═══════════════════════════════════════════
print('🔓 STAGE 2: Full model — fine-tuning all layers with lower LR')
stage1_best = f'{MODELS_DIR}/runs/posecoach_stage1/weights/best.pt'
model2 = YOLO(stage1_best)

model2.train(
    data=yaml_path,
    epochs=30,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project=f'{MODELS_DIR}/runs',
    name='posecoach_v1',
    exist_ok=True,
    freeze=None,         # unfreeze everything
    optimizer='AdamW',
    lr0=0.0005,          # lower LR to preserve backbone features
    lrf=0.01,
    warmup_epochs=1,
    cos_lr=True,
    patience=15,
    save=True,
    save_period=5,
    pose=12.0,
    mosaic=0.5,
    verbose=True,
)
print('\n✅ Two-stage training complete!')

## ✅ Step 10 — Save Final Weights & Evaluate

In [ ]:
import shutil

best_weights = f'{MODELS_DIR}/runs/posecoach_v1/weights/best.pt'
final_path = f'{MODELS_DIR}/yolo_posecoach_v1.pt'

if os.path.exists(best_weights):
    shutil.copy(best_weights, final_path)
    size_mb = os.path.getsize(final_path) / 1e6
    print(f'✅ Weights saved: {final_path} ({size_mb:.1f} MB)')
else:
    print('❌ best.pt not found — check training logs above')

finetuned = YOLO(final_path)
metrics = finetuned.val(data=yaml_path, split='val', verbose=False)

final_results = {
    'model': 'yolo26n-pose-posecoach-v1',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'weights_path': final_path,
    'pose_map50': float(metrics.pose.map50) if hasattr(metrics, 'pose') else 0.0,
    'pose_map': float(metrics.pose.map) if hasattr(metrics, 'pose') else 0.0,
    'thesis_gate_passed': (float(metrics.pose.map50) if hasattr(metrics, 'pose') else 0) >= 0.75
}

with open(f'{DRIVE_ROOT}/data/eval/yolo_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"\n📊 Final OKS-mAP@0.50: {final_results['pose_map50']:.4f}")
print(f"   Thesis gate (>= 0.75): {'✅ PASSED' if final_results['thesis_gate_passed'] else '❌ FAILED — train more epochs'}")
print(f"\n=== NEXT STEPS ===")
print(f'1. Download weights from Drive: {final_path}')
print('2. Place in local: posecoach/models/yolo_posecoach_v1.pt')
print('3. Update .env: MODEL_PATH=models/yolo_posecoach_v1.pt')
print('4. Proceed to P02 — Infrastructure')